# Encoding Technique 5: Target Encoding

**Dataset:** `Loan_Default.csv`

**When to use:** For **nominal** features with **high cardinality**, especially in regression/classification tasks where you want to capture the relationship between a category and the target variable.

**Key concept:** Each category is replaced by the **mean of the target variable** for that category. For example, if the `loan_amount` mean for `'North'` region is `285,000`, then every `'North'` entry in the `Region` column is replaced by `285000`.

---

### Step 1: Setup, Data Loading & Prep

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
import category_encoders as ce

# Load data
df = pd.read_csv('../../data/raw/Loan_Default.csv')
df.drop(['ID', 'year'], axis=1, inplace=True)

categorical_features = df.select_dtypes(include=['object']).columns.tolist()
Ordinal_features = ['age']
Nominal_features = categorical_features.copy()
Nominal_features.remove('age')

# Encode the ordinal feature first
enc = OrdinalEncoder()
df[Ordinal_features] = enc.fit_transform(df[Ordinal_features])

print(f'Shape before Target Encoding: {df.shape}')
df.head()

### Step 2: Apply Target Encoding

The target variable used is `loan_amount`. Notice that `fit_transform` takes **both X (features) and y (target)** — this is what makes it a *supervised* encoding method.

In [ ]:
df_target = df.copy()

# Configure the Target Encoder
Target_encoder = ce.TargetEncoder(cols=Nominal_features)

# Separate numerical and categorical features
df_target_numerical = df_target.drop(Nominal_features, axis=1)

# fit_transform requires the TARGET variable (y)
df_target_categorical = Target_encoder.fit_transform(
    df_target[Nominal_features],
    df_target['loan_amount']   # <-- the target variable
)

# Reassemble
df_target = pd.concat([df_target_numerical, df_target_categorical], axis=1)

print(f'Shape after Target Encoding: {df_target.shape}')
df_target.head()

### Step 3: Inspect the Encoded Values

The nominal columns now contain **continuous mean values** of `loan_amount` per category, instead of string labels.

In [ ]:
# Compare: original Gender categories → encoded mean loan_amount values
print('Target-encoded Gender unique values:')
print(df_target['Gender'].unique())

### Key Observation

Each category is now a **mean of the target**. This approach is extremely compact but requires careful handling (cross-validation) to prevent data leakage in production models.